# Задача 1. Compare

## Объяснение решения

**Merge sort.** Делим диапазон на две половины и сортируем их рекурсивно. При слиянии сравниваем первые ещё не взятые элементы половин и записываем меньший в общий буфер. Если максимум левой половины уже не больше минимума правой, диапазон отсортирован и слияние можно пропустить. При равенстве берём элемент слева, поэтому сортировка стабильна.

**Quicksort.** Берём опорный элемент из середины диапазона. За один проход переставляем значения так, чтобы слева были элементы меньше опорного, в середине — равные ему, справа — большие. Затем сортируем крайние части. Средняя часть уже стоит на своём месте, даже если в ней много элементов. Рекурсия обрабатывает меньшую часть; на большей хвостовой вызов заменён циклом, чтобы ограничить глубину стека.

Каждая функция возвращает новый отсортированный список. Декоратор `timed` оборачивает вызов двумя чтениями `perf_counter` и возвращает пару «результат, секунды». `wraps` сохраняет имя и документацию исходной функции.

## Реализация

In [1]:
from functools import wraps
from random import Random
from statistics import median
from time import perf_counter
from typing import Callable, ParamSpec, TypeVar


P = ParamSpec("P")
R = TypeVar("R")


def timed(function: Callable[P, R]) -> Callable[P, tuple[R, float]]:
    """Возвращает исходный результат и время выполнения в секундах."""

    @wraps(function)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> tuple[R, float]:
        start = perf_counter()
        result = function(*args, **kwargs)
        return result, perf_counter() - start

    return wrapper


def merge_sort(values: list[int]) -> list[int]:
    """Сортирует копию списка рекурсивным слиянием."""
    items = values.copy()
    buffer = items.copy()

    def sort(lo: int, hi: int) -> None:
        if hi - lo < 2:
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid, hi)
        # Обе половины уже отсортированы. Проверяем границу между ними.
        if items[mid - 1] <= items[mid]:
            return

        i, j = lo, mid
        for k in range(lo, hi):
            if j == hi or (i < mid and items[i] <= items[j]):
                buffer[k] = items[i]
                i += 1
            else:
                buffer[k] = items[j]
                j += 1
        items[lo:hi] = buffer[lo:hi]

    sort(0, len(items))
    return items


def quick_sort(values: list[int]) -> list[int]:
    """Сортирует копию списка рекурсивным quicksort с трёхчастным разбиением."""
    items = values.copy()

    def sort(lo: int, hi: int) -> None:
        while hi - lo >= 2:
            pivot = items[(lo + hi) // 2]
            less, cursor, greater = lo, lo, hi
            while cursor < greater:
                if items[cursor] < pivot:
                    items[less], items[cursor] = items[cursor], items[less]
                    less += 1
                    cursor += 1
                elif items[cursor] > pivot:
                    greater -= 1
                    items[cursor], items[greater] = items[greater], items[cursor]
                else:
                    cursor += 1

            # Равные pivot больше не трогаем. Рекурсия — на меньшей части;
            # хвостовую рекурсию на большей заменяем следующей итерацией.
            if less - lo < hi - greater:
                sort(lo, less)
                lo = greater
            else:
                sort(greater, hi)
                hi = less

    sort(0, len(items))
    return items


def benchmark(size: int = 60_000, repeats: int = 7) -> dict[str, tuple[float, float]]:
    """Печатает медианы (merge sort, quicksort) в секундах на трёх входах."""
    if size < 1 or repeats < 1:
        raise ValueError("size и repeats должны быть положительными")

    rng = Random(2026)
    shuffled = list(range(size))
    rng.shuffle(shuffled)
    cases = {
        "Перемешанные уникальные": shuffled,
        "Уже отсортированные": list(range(size)),
        "Случайные, 8 значений": [rng.randrange(8) for _ in range(size)],
    }
    methods = (timed(merge_sort), timed(quick_sort))
    results = {}
    print(f"n={size:,}; повторов: {repeats}; медиана в миллисекундах")
    print(f"{'Вход':28s} {'merge sort':>12s} {'quicksort':>12s} {'merge/quick':>12s}")

    for name, values in cases.items():
        expected = sorted(values)
        samples: tuple[list[float], list[float]] = ([], [])
        # Один прогрев для каждого алгоритма. Генерация входа и проверки
        # результата не входят в измеряемый интервал.
        for method in methods:
            method(values)
        for repeat in range(repeats):
            # Чередуем порядок, чтобы первый запуск не давал фору одному методу.
            for index in (0, 1) if repeat % 2 == 0 else (1, 0):
                result, elapsed = methods[index](values)
                if result != expected:
                    raise ValueError(f"Сортировка вернула неверный результат: {name}")
                samples[index].append(elapsed)

        merge_time, quick_time = map(median, samples)
        results[name] = (merge_time, quick_time)
        ratio = f"{merge_time / quick_time:.2f}×" if quick_time else "—"
        print(f"{name:28s} {merge_time * 1_000:12.2f} {quick_time * 1_000:12.2f} {ratio:>12s}")

    return results

## Визуализация работы алгоритма

Вход: `[7, 2, 7, 4, 1, 2, 5]`.

| Шаг | Merge sort |
| --- | --- |
| 1 | Делим на `[7, 2, 7]` и `[4, 1, 2, 5]`. |
| 2 | Слева делим на `[7]` и `[2, 7]`. В паре `[2, 7]` слияние можно пропустить; сливаем её с `[7]` в `[2, 7, 7]`. |
| 3 | Справа `[4, 1]` превращается в `[1, 4]`, а `[2, 5]` уже отсортирована. Сливаем в `[1, 2, 4, 5]`. |
| 4 | Из двух половин по очереди берём `1, 2, 2, 4, 5, 7, 7`. При равных двойках первой идёт двойка из левой половины. |

| Шаг | Quicksort |
| --- | --- |
| 1 | Опорный элемент — `4`. После разбиения получаем `[2, 2, 1] \| [4] \| [7, 5, 7]`. |
| 2 | Обе крайние части длины 3. По условию в коде первой рекурсивно сортируется правая. Опорный `5` даёт группы `[] \| [5] \| [7, 7]`. |
| 3 | В `[7, 7]` оба элемента попадают в среднюю группу. Правый диапазон отсортирован: `[5, 7, 7]`. |
| 4 | Цикл продолжает на левой части. Опорный `2` даёт `[1] \| [2, 2] \| []`. Все части уже находятся в исходном списке в нужном порядке: `[1, 2, 2, 4, 5, 7, 7]`. |

## Оценка сложности

| Реализация | Время | Память |
| --- | --- | --- |
| Merge sort | `O(n log n)` в среднем на случайной перестановке и в худшем случае: на каждом из `O(log n)` уровней суммарно сливаем до `n` элементов. На отсортированном входе `O(n)`: слияния пропускаются, остаются копирование и вызовы. | `O(n)` под буфер и временные срезы, стек `O(log n)`. |
| Quicksort | `O(n log n)` при сбалансированных разбиениях и в среднем на случайной перестановке разных элементов. При последовательных разбиениях «один элемент и почти весь диапазон» — `O(n²)`. Если все значения равны, достаточно одного прохода: `O(n)`. | Стек `O(log n)` даже при плохих разбиениях: рекурсия всегда получает не больше половины текущего диапазона. Разбиение на месте требует `O(1)` памяти на один вызов. Копия входа для результата — `O(n)`. |
| Декоратор `timed` | `O(1)` для замеряемых сортировок с одним аргументом: два показания таймера и упаковка результата. Передача произвольного числа аргументов занимает до `O(p)`, где `p` — их количество. | `O(1)` сверх результата функции при фиксированном числе аргументов; до `O(p)` в общем случае. |

## Методика замеров

Тесты проверяют пустой список, одиночный элемент, отсортированные и обратные последовательности, отрицательные числа, повторы и случайные списки. Дополнительно проверяем сохранение исходного списка и поведение декоратора. Тесты не требуют, чтобы один алгоритм работал быстрее другого.

Для замеров берём по 60 000 элементов: случайную перестановку уникальных чисел, отсортированные числа и случайные числа из восьми значений. Обе сортировки получают одинаковые входы. Для каждого случая выполняем прогрев, делаем семь замеров с чередованием порядка вызовов и берём медиану. Подготовка данных и сверка с `sorted` идут вне таймера, а создание копии включено в замер для обеих сортировок.

В контрольном запуске на Python 3.12 merge sort обработал отсортированный вход за 7,12 мс, quicksort — за 51,24 мс. На данных из восьми значений порядок поменялся: 53,90 мс у merge sort и 8,33 мс у quicksort. В первом случае помогает пропуск слияний, во втором — удаление из дальнейшей работы целых групп равных элементов. Эти числа относятся к данному запуску; значения в другой среде будут отличаться.

## Результаты замеров

In [2]:
measurements = benchmark()

n=60,000; повторов: 7; медиана в миллисекундах
Вход                           merge sort    quicksort  merge/quick


Перемешанные уникальные             65.77        72.75        0.90×


Уже отсортированные                  7.69        56.38        0.14×


Случайные, 8 значений               53.69         8.19        6.56×


## Тесты

In [3]:
import unittest

from collections import Counter
from random import Random
from unittest import TestCase
from unittest.mock import Mock, patch



class SortTests(TestCase):
    def test_typical_and_boundary_cases(self) -> None:
        cases = (
            ([], []),
            ([42], [42]),
            ([2, 1], [1, 2]),
            ([-3, -1, 0, 2], [-3, -1, 0, 2]),
            ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
            ([7, 2, 7, 4, 1, 2, 5], [1, 2, 2, 4, 5, 7, 7]),
            ([2, 0, -3, 2, -3], [-3, -3, 0, 2, 2]),
            ([11] * 2000, [11] * 2000),
        )
        for sort in (merge_sort, quick_sort):
            for values, expected in cases:
                with self.subTest(algorithm=sort.__name__, size=len(values)):
                    original = values.copy()
                    self.assertEqual(sort(values), expected)
                    self.assertEqual(values, original)

    def test_randomised_cases_match_reference(self) -> None:
        rng = Random(2026)
        for size in (2, 17, 64, 257):
            values = [rng.randrange(-size, size) for _ in range(size)]
            for sort in (merge_sort, quick_sort):
                with self.subTest(algorithm=sort.__name__, size=size):
                    self.assertEqual(sort(values), sorted(values))

    def test_mostly_one_value_preserves_all_counts(self) -> None:
        values = [7] * 10_000 + [-2, 8, -2, 8, 9]
        Random(2026).shuffle(values)
        for sort in (merge_sort, quick_sort):
            with self.subTest(algorithm=sort.__name__):
                result = sort(values)
                self.assertEqual(Counter(result), Counter(values))
                self.assertTrue(all(left <= right for left, right in zip(result, result[1:])))


class TimingTests(TestCase):
    def test_wrapper_preserves_result_arguments_and_name(self) -> None:
        @timed
        def combine(left: int, *, right: int) -> tuple[int, int]:
            return left, right

        # Замеры в тесте детерминированы; скорость машины не влияет на результат.
        timer = Mock(side_effect=[10.0, 10.25])
        with patch.dict(timed.__globals__, {"perf_counter": timer}):
            result, seconds = combine(1, right=4)

        self.assertEqual(result, (1, 4))
        self.assertEqual(seconds, 0.25)
        self.assertEqual(combine.__name__, "combine")

suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(SortTests))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TimingTests))
report = unittest.TextTestRunner(verbosity=2).run(suite)
assert report.wasSuccessful(), 'Не все тесты прошли'

test_mostly_one_value_preserves_all_counts (__main__.SortTests.test_mostly_one_value_preserves_all_counts) ... 

ok


test_randomised_cases_match_reference (__main__.SortTests.test_randomised_cases_match_reference) ... 

ok


test_typical_and_boundary_cases (__main__.SortTests.test_typical_and_boundary_cases) ... 

ok


test_wrapper_preserves_result_arguments_and_name (__main__.TimingTests.test_wrapper_preserves_result_arguments_and_name) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.007s

OK
